## **9: Pipelines in sklearn**

A Pipeline is a way to connect multiple steps (like preprocessing + model) into one flow

**Simple Intuition** <br>
Instead of doing this manually ❌
1. scale data
2. train model
3. predict <br>

Pipeline does everything in one line ✅

**Why we use Pipeline?**
* Avoid repeating code
* Keeps workflow clean
* Prevents data leakage
* Easy to use with cross-validation

**Basic Syntax**

```bash
pipe = Pipeline([
    ('step_name1', transformer1),
    ('step_name2', transformer2),
    ('model', model)
])
```

**Step names are labels given to each step inside a Pipeline**


**Methods**
* pipe.fit(X_train, y_train)
* pipe.predict(X_test)
* pipe.score(X_test, y_test)

**Data flow(Important):**  X → scaler → model → output


In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load dataset
X, y = load_iris(return_X_y=True)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Step 1: Scale data
    ('model', LogisticRegression())    # Step 2: Train model
])

# Train
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Accuracy
print("Accuracy:", pipeline.score(X_test, y_test))

Accuracy: 1.0


### **Pipeline + GridSearchCV**

In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

# 🔹 Load dataset
X, y = load_iris(return_X_y=True)

# 🔹 Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🔹 Create Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),          # Step 1: Scaling
    ('model', LogisticRegression(max_iter=200))  # Step 2: Model
])

# 🔹 Define Hyperparameters
params = {
    'model__C': [0.1, 1, 10],        # Regularization
    'model__solver': ['lbfgs', 'liblinear']
}

# 🔹 GridSearchCV
grid = GridSearchCV(pipeline, param_grid=params, cv=5)

# 🔹 Train
grid.fit(X_train, y_train)

# 🔹 Best Parameters
print("Best Parameters:", grid.best_params_)

# 🔹 Prediction
y_pred = grid.predict(X_test)

# 🔹 Evaluation
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best Parameters: {'model__C': 1, 'model__solver': 'lbfgs'}

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



C:\Users\shiva\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
15 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\shiva\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\shiva\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\shiva\AppData\Roaming\Python\Python313\site-p

### **ColumnTransformer (IMPORTANT)**

ColumnTransformer is used to apply different preprocessing to different columns in a dataset.

**Simple Intuition**
* Suppose your dataset has:
    * Age (numeric)
    * Salary (numeric)
    * Gender (categorical)
* You want:
    * Scale numbers ✅
    * Encode categories ✅

👉 ColumnTransformer lets you do this in one step

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 🔹 Sample Dataset
data = {
    'Age': [25, 30, 35, 40, 28],
    'Salary': [50000, 60000, 70000, 80000, 52000],
    'Gender': ['Male', 'Female', 'Female', 'Male', 'Male'],
    'Purchased': [0, 1, 1, 0, 0]
}

df = pd.DataFrame(data)

# 🔹 Features & Target
X = df[['Age', 'Salary', 'Gender']]
y = df['Purchased']

# 🔹 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🔹 ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['Age', 'Salary']),
    ('cat', OneHotEncoder(), ['Gender'])
])

# 🔹 Pipeline
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression())
])

# 🔹 Train
pipeline.fit(X_train, y_train)

# 🔹 Predict
y_pred = pipeline.predict(X_test)

# 🔹 Accuracy
print("Accuracy:", pipeline.score(X_test, y_test))

Accuracy: 0.0


### **make_pipeline in sklearn**

👉 make_pipeline is a shortcut to create a Pipeline without giving step names manually.

**Using Pipeline**
```bash
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])
```

**Using make_pipeline**
```bash
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)
```

**make_pipeline automatically assigns names:**
* StandardScaler() → standardscaler
* LogisticRegression() → logisticregression